# SeqTrainer + Nucleotide Transformer v2

This notebook shows a package-first workflow for promoter **classification** and **regression** with SeqTrainer and Nucleotide Transformer v2.

## 1) Setup

Install torch extras if needed:
```bash
pip install -e './[torch]'
```

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

from seqtrainer.torch import build_finetune_config, nucleotide_transformer_v2

## 2) Load promoter data

Expected columns:
- `sequence`: promoter DNA sequence
- `label`: binary class label for classification
- `activity`: continuous value for regression

In [ ]:
df = pd.read_csv('../data/dataset_builder/processed_dataset.csv')

if 'sequence' not in df.columns:
    seq_col = [c for c in df.columns if 'sequence' in c.lower()][0]
    df = df.rename(columns={seq_col: 'sequence'})

if 'label' not in df.columns:
    y_source = 'expression' if 'expression' in df.columns else df.columns[-1]
    threshold = df[y_source].quantile(0.70)
    df['label'] = (df[y_source] >= threshold).astype(int)

if 'activity' not in df.columns:
    y_source = 'expression' if 'expression' in df.columns else df.columns[-1]
    df['activity'] = df[y_source].astype(float)

df = df[['sequence', 'label', 'activity']].dropna().reset_index(drop=True)
df.head()

## 3) Build SeqTrainer backbone config

In [ ]:
backbone = nucleotide_transformer_v2(variant='500m-multi-species')
finetune_cfg = build_finetune_config(
    backbone=backbone.model_id,
    head='classification_or_regression',
    learning_rate=2e-5,
    epochs=3,
)
backbone, finetune_cfg

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(backbone.tokenizer_id, trust_remote_code=backbone.trust_remote_code)
encoder = AutoModel.from_pretrained(backbone.model_id, trust_remote_code=backbone.trust_remote_code)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = encoder.to(device)

## 4) Define dataset and heads

In [ ]:
class PromoterDataset(Dataset):
    def __init__(self, frame, target_col):
        self.frame = frame.reset_index(drop=True)
        self.target_col = target_col

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        return row['sequence'], float(row[self.target_col])


def collate_batch(batch, max_length=256):
    seqs, ys = zip(*batch)
    tokens = tokenizer(
        list(seqs),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    ys = torch.tensor(ys, dtype=torch.float32)
    return tokens, ys


class ClassificationHead(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, hidden_size // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class RegressionHead(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, hidden_size // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

## 5) Helpers for training/evaluation

In [ ]:
def pooled_embedding(tokens):
    tokens = {k: v.to(device) for k, v in tokens.items()}
    outputs = encoder(**tokens)
    return outputs.last_hidden_state[:, 0, :]


def run_epoch(head, loader, optimizer, loss_fn):
    head.train()
    encoder.train()
    losses = []
    for tokens, y in loader:
        y = y.to(device)
        optimizer.zero_grad()
        z = pooled_embedding(tokens)
        pred = head(z)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def predict(head, loader):
    head.eval()
    encoder.eval()
    preds, ys = [], []
    for tokens, y in loader:
        y = y.to(device)
        z = pooled_embedding(tokens)
        p = head(z)
        preds.extend(p.detach().cpu().numpy().tolist())
        ys.extend(y.detach().cpu().numpy().tolist())
    return np.array(preds), np.array(ys)

## 6) Promoter classification example

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_cls = DataLoader(PromoterDataset(train_df, 'label'), batch_size=8, shuffle=True, collate_fn=collate_batch)
test_cls = DataLoader(PromoterDataset(test_df, 'label'), batch_size=8, shuffle=False, collate_fn=collate_batch)

cls_head = ClassificationHead(encoder.config.hidden_size).to(device)
cls_opt = torch.optim.AdamW(list(encoder.parameters()) + list(cls_head.parameters()), lr=finetune_cfg['learning_rate'])
cls_loss = torch.nn.BCEWithLogitsLoss()

for epoch in range(finetune_cfg['epochs']):
    loss = run_epoch(cls_head, train_cls, cls_opt, cls_loss)
    print(f'Classification epoch {epoch+1}: loss={loss:.4f}')

logits, y_true = predict(cls_head, test_cls)
y_prob = 1 / (1 + np.exp(-logits))
y_pred = (y_prob >= 0.5).astype(int)

print('Accuracy:', accuracy_score(y_true, y_pred))
print('F1:', f1_score(y_true, y_pred))

## 7) Promoter regression example

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_reg = DataLoader(PromoterDataset(train_df, 'activity'), batch_size=8, shuffle=True, collate_fn=collate_batch)
test_reg = DataLoader(PromoterDataset(test_df, 'activity'), batch_size=8, shuffle=False, collate_fn=collate_batch)

reg_head = RegressionHead(encoder.config.hidden_size).to(device)
reg_opt = torch.optim.AdamW(list(encoder.parameters()) + list(reg_head.parameters()), lr=finetune_cfg['learning_rate'])
reg_loss = torch.nn.MSELoss()

for epoch in range(finetune_cfg['epochs']):
    loss = run_epoch(reg_head, train_reg, reg_opt, reg_loss)
    print(f'Regression epoch {epoch+1}: loss={loss:.4f}')

pred, y_true = predict(reg_head, test_reg)
print('RMSE:', mean_squared_error(y_true, pred, squared=False))
print('R2:', r2_score(y_true, pred))